<a href="https://colab.research.google.com/github/zencolab/colab/blob/main/comfyui_333.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 一、安装 ComfyUI 与依赖环境 (图像与视频双修版)
# ==========================================
import os

# 1. 切换到主目录
%cd /content

# 智能判断：防止重复克隆导致的 fatal 报错，并确保是最新版 (自带EmptyLatentVideo节点)
if not os.path.exists("/content/ComfyUI"):
    print("⏳ 正在拉取 ComfyUI 官方最新源码...")
    !git clone https://github.com/comfyanonymous/ComfyUI
else:
    print("✨ 检测到 ComfyUI 已存在，正在拉取最新核心更新...")
    %cd /content/ComfyUI
    !git pull

# 确保当前路径在 ComfyUI 内
%cd /content/ComfyUI

# 2. 【核心升级】安装带有 CUDA 13.0 深度优化的 PyTorch (使用系统默认版，已注释)
#print("⏳ 正在安装 PyTorch 及其核心依赖 (基于 cu130)...")
#!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

# 3. 安装 ComfyUI 官方运行依赖
print("⏳ 正在安装 ComfyUI 官方运行依赖...")
!pip install -q -r requirements.txt

# 4. 【关键配置】独立安装 huggingface_hub 和 hf_transfer，开启满速下载
print("⏳ 正在配置 Hugging Face 高速下载环境...")
!pip install -q huggingface_hub hf_transfer

# 5. 【重磅新增】安装业内必备的高级视频处理插件包 (VideoHelperSuite)
print("⏳ 正在检查并安装高级视频处理插件包...")
%cd /content/ComfyUI/custom_nodes
if not os.path.exists("ComfyUI-VideoHelperSuite"):
    !git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
    %cd ComfyUI-VideoHelperSuite
    !pip install -q -r requirements.txt
    print("✅ 视频插件包 (VideoHelperSuite) 安装成功！")
else:
    print("✨ 视频插件包已存在，正在检查更新...")
    %cd ComfyUI-VideoHelperSuite
    !git pull

# 恢复当前路径到 ComfyUI 根目录，以免影响后续操作
%cd /content/ComfyUI

print("\n🎉 第一阶段完成：ComfyUI 核心升级完毕，图像与视频双修环境已就绪！")

In [ ]:
# ==========================================
# 二、终极模型下载列表 (Flux2 图像 + LTX-2.3 视频双擎最强版)
# ==========================================
import os
from google.colab import userdata
from huggingface_hub import hf_hub_download

# 1. ⚠️必须在导入任何 HF 库之前，先开启高速满速下载魔法！
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# 2. 从 Colab Secrets 中静默读取并写入 HF Token
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = HF_TOKEN
except userdata.SecretNotFoundError:
    print("❌ 错误：请先在左侧 '🔑 Secrets' 中添加名为 'HF_TOKEN' 的密钥！")
    raise

# 3. 灵活配置下载任务列表
downloads = [
    # -------------------【第一部分：Flux 图像生态】-------------------
    # 1. Flux 主模型
    {
        "repo_id": "stronman/flux2-dev",
        "filename": "split_files/diffusion_models/flux2_dev_fp8mixed.safetensors",
        "local_dir": "/content/ComfyUI/models/unet"
    },
    # 2. Flux Turbo LoRA (加速)
    {
        "repo_id": "stronman/flux2-dev",
        "filename": "split_files/loras/Flux2TurboComfyv2.safetensors",
        "local_dir": "/content/ComfyUI/models/loras"
    },
    # 3. Flux 文本编码器 (Mistral)
    {
        "repo_id": "stronman/flux2-dev",
        "filename": "split_files/text_encoders/mistral_3_small_flux2_bf16.safetensors",
        "local_dir": "/content/ComfyUI/models/clip"
    },
    # 4. Flux VAE 解码器
    {
        "repo_id": "stronman/flux2-dev",
        "filename": "split_files/vae/flux2-vae.safetensors",
        "local_dir": "/content/ComfyUI/models/vae"
    },
    # 5. Flux 全能 ControlNet (Union Pro 2.0 姿势/线稿控制)
    {
        "repo_id": "Shakker-Labs/FLUX.1-dev-ControlNet-Union-Pro-2.0",
        "filename": "diffusion_pytorch_model.safetensors",
        "local_dir": "/content/ComfyUI/models/controlnet"
    },
    # 6. Flux NSFW Slider LoRA (风格微调)
    {
        "repo_id": "xey/sldr_flux_nsfw_v2-studio",
        "filename": "sldr_flux_nsfw_v2-studio.safetensors",
        "local_dir": "/content/ComfyUI/models/loras"
    },

    # -------------------【第二部分：LTX-2.3 视频生态】-------------------
    # 7. LTX-2.3 视频主模型
    {
        "repo_id": "stronman/LTX-2.3",
        "filename": "ltx-2.3-22b-dev.safetensors",
        "local_dir": "/content/ComfyUI/models/unet"
    },
    # 8. 【已修正】LTX-2.3 官方原配翻译官 (Gemma 3 12B FP8版)
    {
        "repo_id": "Comfy-Org/ltx-2",
        "filename": "split_files/text_encoders/gemma_3_12B_it_fp8_scaled.safetensors",
        "local_dir": "/content/ComfyUI/models/clip"
    },
    # 9. 通用辅助文本编码器 CLIP-L
    {
        "repo_id": "comfyanonymous/flux_text_encoders",
        "filename": "clip_l.safetensors",
        "local_dir": "/content/ComfyUI/models/clip"
    },
    # 10. LTX-2.3 专属 VAE 解码器 (稳定镜像源)
    {
        "repo_id": "calcuis/ltxv-gguf",
        "filename": "ltx-video-vae.safetensors",
        "local_dir": "/content/ComfyUI/models/vae"
    }
]

print(f"\n🚀 开始满速同步指定的 {len(downloads)} 个模型文件...\n")

# 4. 执行循环下载
for i, task in enumerate(downloads, 1):
    repo = task["repo_id"]
    file_path = task["filename"]
    target_dir = task["local_dir"]
    file_name = file_path.split('/')[-1]

    print(f"[{i}/{len(downloads)}] 正在从 {repo} 同步: {file_name}")
    try:
        hf_hub_download(
            repo_id=repo,
            filename=file_path,
            local_dir=target_dir,
            repo_type="model"
        )
        print(f"✅ 已完成并存放到: {target_dir}\n")
    except Exception as e:
        print(f"❌ 下载失败 ({file_name}): {e}\n")

print("🎉 恭喜！融入 Gemma 3 官方大脑的 10 个核心组件已全部同步完毕！")

In [ ]:
# ==========================================
# 三、内网穿透配置 (只需运行一次)
# ==========================================
import os
from google.colab import userdata

# 1. 读取 Colab Secrets
try:
    VPS_IP = userdata.get('VPS_IP')
    FRP_TOKEN = userdata.get('FRP_TOKEN')
except Exception as e:
    print("❌ 错误：无法读取 Secrets！请检查左侧是否填写并开启了 'VPS_IP' 和 'FRP_TOKEN'。")
    raise e

# 2. 下载并解压 FRP 客户端
%cd /content
if not os.path.exists("/content/frp_0.56.0_linux_amd64"):
    print("⏳ 正在下载并解压 FRP 客户端...")
    !wget -q https://github.com/fatedier/frp/releases/download/v0.56.0/frp_0.56.0_linux_amd64.tar.gz
    !tar -xzf frp_0.56.0_linux_amd64.tar.gz
    !rm frp_0.56.0_linux_amd64.tar.gz

# 3. 动态生成并写入配置文件
print("⏳ 正在生成 frpc.toml 配置文件...")
frpc_config_content = f"""
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui_web"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8080
"""

with open("/content/frp_0.56.0_linux_amd64/frpc.toml", "w", encoding="utf-8") as f:
    f.write(frpc_config_content.strip())

print("✅ FRP 配置文件生成完毕！")

In [ ]:
# ==========================================
# 四、启动 FRP 和 ComfyUI (重启界面时运行)
# ==========================================
import subprocess
import threading
from google.colab import userdata

# 获取 IP 仅为了在控制台友好地打印访问链接
try:
    VPS_IP = userdata.get('VPS_IP')
except:
    VPS_IP = "你的公网IP"

# 1. 在后台新线程启动 FRP (读取刚才生成的配置文件)
def start_frpc():
    subprocess.run(["/content/frp_0.56.0_linux_amd64/frpc", "-c", "/content/frp_0.56.0_linux_amd64/frpc.toml"])

print("⏳ 正在后台唤起 FRP 穿透服务...")
threading.Thread(target=start_frpc, daemon=True).start()

print("============================================================")
print("✅ FRP 穿透已就绪！")
print(f"👉 启动完成后，请通过浏览器访问: http://{VPS_IP}:8080")
print("============================================================\n")

# 2. 启动 ComfyUI 主程序
print("⏳ 正在启动 ComfyUI 主进程...\n")
%cd /content/ComfyUI
!python main.py --dont-print-server

In [ ]:
# ==========================================
# 暴力强制升级 ComfyUI 核心 (解决更新卡死)
# ==========================================
import os

print("⏳ 正在强制清理本地冲突，并同步官方最新核心代码...")
%cd /content/ComfyUI

# 1. 放弃所有本地可能的冲突修改，强制重置
!git reset --hard HEAD

# 2. 强制拉取官方最新版本 (这步将真正为你带来 EmptyLatentVideo 节点)
!git pull

print("\n🎉 强制升级完毕！")

In [ ]:
# ==========================================
# 单点补全：下载官方原配 Gemma 3 (12B) 文本编码器
# ==========================================
import os
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("⏳ 正在连接 Comfy-Org 官方仓库下载 Gemma 3 (FP8高清版)...")

try:
    hf_hub_download(
        repo_id="Comfy-Org/ltx-2",
        filename="split_files/text_encoders/gemma_3_12B_it_fp8_scaled.safetensors",
        local_dir="/content/ComfyUI/models/clip",
        repo_type="model"
    )
    print("✅ 成功！原配 Gemma 3 翻译官已顺利入库！")
    print("🎉 恭喜！你的 10 大核心组件终于真·100% 同步完毕！")
except Exception as e:
    print(f"❌ 下载依然失败: {e}")